In [1]:
%load_ext autoreload 
#hopefully this will reload the modules when they are changed specifically when i change the plotting modules
%autoreload 2

In [2]:
# from nxs_analysis_tools import *
from nxs_analysis_tools.datareduction import load_transform, load_data
import matplotlib.pyplot as plt

In [3]:
from nexusformat.nexus import NXdata, nxsetmemory
nxsetmemory(300000)  # Set to 80000 MB or higher

In [5]:
# data = load_data('example_data/pairdistribution_data/vacancies.nxs')
data = load_transform('/home/apoulin/de-lat-to-4431-b_link/nxrefine/Eu5Sn2As6/sample1/Eu5Sn2As6_300.nxs')

data:NXdata
  @axes = ['Qh', 'Qk', 'Ql']
  @signal = 'counts'
  Qh = float64(1041)
    @long_name = 'H (r.l.u.)'
    @scaling_factor = 0.5116183785668582
  Qk = float64(1201)
    @long_name = 'K (r.l.u.)'
    @scaling_factor = 0.45069832201273835
  Ql = float64(1001)
    @long_name = 'L (r.l.u.)'
    @scaling_factor = 1.4949642644791898
  counts = float32(1041x1201x1001)


In [ ]:
# data = load_data('example_data/pairdistribution_data/vacancies.nxs')
dataBig = load_transform('/home/apoulin/de-lat-to-4431-b_link/nxrefine/FeSc2S4/sample1/FeSc2S4_300.nxs')

In [6]:
import os, psutil
from nexusformat.nexus import NXdata

def rss_gb():
    return psutil.Process(os.getpid()).memory_info().rss / (1024**3)

print("type(data):", type(data))
assert isinstance(data, NXdata)
print("RSS (GB):", rss_gb())

sig = data.nxsignal          # NXfield (often lazily backed)
axes = data.nxaxes           # list of NXfield axes
print("signal type:", type(sig))
print("signal shape:", getattr(sig, "shape", None))
print("signal dtype:", getattr(sig, "dtype", None))
print("axes:", [getattr(a, "name", None) for a in axes])
print("RSS (GB) after refs:", rss_gb())

# IMPORTANT: avoid sig.nxdata / sig.nxvalue / np.asarray(sig) here

type(data): <class 'nexusformat.nexus.tree.NXdata'>
RSS (GB): 4.884532928466797
signal type: <class 'nexusformat.nexus.tree.NXfield'>
signal shape: (1041, 1201, 1001)
signal dtype: float32
axes: [None, None, None]
RSS (GB) after refs: 4.884700775146484


In [7]:
%whos

Variable         Type        Data/Info
--------------------------------------
NXdata           type        <class 'nexusformat.nexus.tree.NXdata'>
axes             list        n=3
data             NXdata      data
load_data        function    <function load_data at 0x7efdfc2dccc0>
load_transform   function    <function load_transform at 0x7efd3a46dd00>
nxsetmemory      function    <function setmemory at 0x7efd67aa2020>
os               module      <module 'os' (frozen)>
plt              module      <module 'matplotlib.pyplo<...>es/matplotlib/pyplot.py'>
psutil           module      <module 'psutil' from '/h<...>ages/psutil/__init__.py'>
rss_gb           function    <function rss_gb at 0x7efd35f32660>
sig              NXfield     [[[0. 0. 0. ... 0. 0. 0.]<...>[0. 0. 0. ... 0. 0. 0.]]]


In [8]:
import numpy as np

H, K, L = sig.shape  # assumes (H,K,L)
l_idx = L // 2

# This should only read a (H,K) plane, not the full 3D volume
hk_plane = sig[:, :, l_idx]     # returns a NumPy array for that slice
print("hk_plane shape:", hk_plane.shape, "dtype:", hk_plane.dtype)
print("RSS (GB) after slice:", rss_gb())

hk_plane shape: (1041, 1201) dtype: float32
RSS (GB) after slice: 4.884727478027344


In [9]:
h0, h1 = 100, 200
k0, k1 = 100, 200
l0, l1 =  40,  60

block = sig[h0:h1, k0:k1, l0:l1]   # only loads that window
print("block shape:", block.shape)
print("RSS (GB) after block:", rss_gb())

block shape: (100, 100, 20)
RSS (GB) after block: 4.884742736816406


In [10]:
import dask.array as da

# Choose chunks that match your access pattern (tune these)
d_sig = da.from_array(sig, chunks=(64, 64, 16))

# Lazy: no data read yet
lazy_plane = d_sig[:, :, l_idx]

# Actually read/compute now:
hk_plane2 = lazy_plane.compute()
print(hk_plane2.shape, hk_plane2.dtype)

(1041, 1201) float32


In [11]:
from dataclasses import dataclass
from typing import Tuple, List, Optional

import dask.array as da
from nexusformat.nexus import NXdata, NXfield

@dataclass(frozen=True)
class LazyNXDataView:
    nx: NXdata
    signal_name: str
    axes_names: Tuple[str, ...]
    chunks: Tuple[int, int, int] = (64, 64, 16)  # tune for your access pattern

    @classmethod
    def from_nxdata(cls, nx: NXdata, chunks=(64, 64, 16)) -> "LazyNXDataView":
        # In your file: @signal='counts', @axes=['Qh','Qk','Ql']
        signal_name = nx.attrs.get("signal", None)
        if not signal_name:
            # fallback: try nxsignal name
            signal_name = getattr(nx.nxsignal, "nxname", None) or "counts"

        axes = nx.attrs.get("axes", [])
        axes_names = tuple(axes) if axes else tuple()

        return cls(nx=nx, signal_name=str(signal_name), axes_names=axes_names, chunks=chunks)

    @property
    def signal(self) -> NXfield:
        return self.nx[self.signal_name]

    @property
    def axes(self) -> List[NXfield]:
        return [self.nx[name] for name in self.axes_names]

    def dask_signal(self) -> da.Array:
        # stays lazy; data read happens on compute()
        return da.from_array(self.signal, chunks=self.chunks)

    # Convenience slice helpers (these read only the requested region)
    def hk_plane(self, l_idx: int):
        return self.signal[:, :, l_idx]

    def window(self, h: slice, k: slice, l: slice):
        return self.signal[h, k, l]

In [12]:
view = LazyNXDataView.from_nxdata(data, chunks=(128, 128, 16))

sig = view.signal                 # NXfield (lazy)
Qh, Qk, Ql = view.axes            # NXfield axes (lazy)

hk = view.hk_plane(l_idx=500)     # reads only one (H,K) plane

# Or keep it lazy with Dask:
d_sig = view.dask_signal()
hk_lazy = d_sig[:, :, 500]
hk2 = hk_lazy.compute()           # triggers I/O for just that plane